# 01 — Signal Exploration

Load synthetic rocket telemetry, visualize time/frequency domains, add noise, filter, compress, and reconstruct.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.data_loader.loader import load_signal
from src.generate_synthetic import create_synthetic_dataset
from src.preprocessing.noise import apply_all_noise
from src.preprocessing.filtering import butterworth_lowpass, moving_average, savgol_filter
from src.compression.fft_compression import compress_fft, decompress_fft
from src.compression.wavelet_compression import compress_wavelet, decompress_wavelet
from src.compression.quantization import compress_quantization, decompress_quantization
from src.metrics.mse import mse, rmse
from src.metrics.snr import snr_db
from src.metrics.compression_ratio import compression_ratio
from src.visualization.plots import plot_signal_overview, plot_comparison

%matplotlib inline

## 1. Generate & Load Signal

In [ ]:
data_path = create_synthetic_dataset()
time, signal, metadata = load_signal(data_path)
fs = metadata['sampling_frequency']
print(f"Loaded {metadata['n_samples']} samples, {metadata['duration']:.1f}s, fs={fs:.0f} Hz")
print(f"Range: [{metadata['min']:.2f}, {metadata['max']:.2f}] m/s²")

## 2. Visualize Time & Frequency Domains

In [ ]:
plot_signal_overview(time, signal, fs, title_prefix="Original Rocket Telemetry");

## 3. Add Realistic Noise

In [ ]:
noisy, noise_info = apply_all_noise(signal, gaussian_level=0.05, impulse_prob=0.005, seed=42)
plot_comparison(time, signal, noisy, "Clean", "Noisy", "Noise Impact");

## 4. Filter Comparison

In [ ]:
filtered_butter = butterworth_lowpass(noisy, fs, cutoff_frequency=15.0)
filtered_ma = moving_average(noisy, window_size=11)
filtered_sg = savgol_filter(noisy, window_length=11, polyorder=3)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for ax, filt, name in zip(axes, [filtered_butter, filtered_ma, filtered_sg],
                          ["Butterworth LP", "Moving Average", "Savitzky-Golay"]):
    ax.plot(time, noisy, alpha=0.3, label="Noisy")
    ax.plot(time, filt, label=name)
    ax.legend()
    ax.set_title(name)
    ax.grid(True, alpha=0.3)
plt.tight_layout();

## 5. FFT Compression Sweep

In [ ]:
working = filtered_butter
keep_rates = [0.01, 0.05, 0.10, 0.25, 0.50]

print(f"{'Keep %':>8} {'Ratio':>8} {'MSE':>12} {'SNR (dB)':>10}")
print("-" * 42)
for rate in keep_rates:
    comp = compress_fft(working, keep_percentage=rate)
    recon = decompress_fft(comp)
    ratio = compression_ratio(working, comp)
    print(f"{rate*100:>7.0f}% {ratio:>7.1f}x {mse(working, recon):>12.6f} {snr_db(working, recon):>10.1f}")

## 6. Wavelet Compression Comparison

In [ ]:
for wavelet in ["haar", "db4", "db8"]:
    comp = compress_wavelet(working, wavelet=wavelet, keep_percentage=0.10)
    recon = decompress_wavelet(comp)
    print(f"{wavelet:>5}: MSE={mse(working, recon):.6f}, SNR={snr_db(working, recon):.1f} dB, ratio={compression_ratio(working, comp):.1f}x")

## 7. Reconstruction Comparison (10% keep)

In [ ]:
comp_fft = compress_fft(working, keep_percentage=0.10)
recon_fft = decompress_fft(comp_fft)
plot_comparison(time, working, recon_fft, "Filtered", "FFT Reconstructed (10%)", "FFT Compression");

## 8. Quantization Comparison

In [ ]:
for bits in [8, 16]:
    comp = compress_quantization(working, bits=bits)
    recon = decompress_quantization(comp)
    print(f"{bits}-bit: MSE={mse(working, recon):.8f}, SNR={snr_db(working, recon):.1f} dB, ratio={compression_ratio(working, comp):.1f}x")